# 1. Event data processing

### Expected outcome

Five related tables: `teams`, `players`, `games`, `event_types`, and `events`, using common event definitions across the source datasets.


## 1.1. Setup and load raw data

In [88]:
import pandas as pd
import numpy as np

from pathlib import Path
import re

pd.set_option('display.max_columns', None)

In [89]:
ROOT_DIR = '..//'
RAW_DATA_DIR = Path(ROOT_DIR) / 'data' / 'raw'

nwhl_df = pd.read_csv(f'{RAW_DATA_DIR}/hackathon_nwhl.csv')
womens_df = pd.read_csv(f'{RAW_DATA_DIR}/hackathon_womens.csv')
olympics_df = pd.read_csv(f'{RAW_DATA_DIR}/pxp_womens_oly_2022_v2.csv')

datasets = {
    'NWHL': nwhl_df,
    'Womens': womens_df,
    'Olympics': olympics_df,
}

## 1.2. Map and standardize source fields

### Column mappings

Map the original columns to the destination columns

#### Legacy (Womens and NWHL)

In [90]:
legacy_columns = {
    "Home Team": "home_team",
    "Away Team": "away_team",
    "Period": "period",
    "Clock": "clock",
    "Home Team Skaters": "home_skaters",
    "Away Team Skaters": "away_skaters",
    "Home Team Goals": "home_goals",
    "Away Team Goals": "away_goals",
    "Team": "team_name",
    "Player": "player_name",
    "Event": "event",
    "X Coordinate": "x",
    "Y Coordinate": "y",
    "Player 2": "player_name_2",
    "X Coordinate 2": "x_2",
    "Y Coordinate 2": "y_2",
    "Detail 1": "detail_1",
    "Detail 2": "detail_2",
    "Detail 3": "detail_3",
    "Detail 4": "detail_4",
}


In [91]:
# Process the clock column to get seconds remaining in the period
def clock_to_seconds(clock_str):
    if pd.isna(clock_str):
        return np.nan
    match = re.match(r"(\d+):(\d+)", clock_str)
    if match:
        minutes, seconds = map(int, match.groups())
        return minutes * 60 + seconds
    else:
        raise ValueError(f"Invalid clock format: {clock_str}")

def map_legacy_columns(raw, source_dataset):
    df = raw.rename(columns=legacy_columns).copy()

    is_home = df["team_name"].eq(df["home_team"])
    is_away = df["team_name"].eq(df["away_team"])

    if (~(is_home | is_away)).any():
        raise ValueError("Some event teams do not match either game team.")

    df["is_home"] = is_home.astype("boolean")

    field_pairs = {
        "team_skaters": ("home_skaters", "away_skaters"),
        "opponent_skaters": ("away_skaters", "home_skaters"),
        "team_goals": ("home_goals", "away_goals"),
        "opponent_goals": ("away_goals", "home_goals"),
    }

    for destination, (home_event, away_event) in field_pairs.items():
        df[destination] = df[home_event].where(is_home, df[away_event])

    df["clock"] = df["clock"].apply(clock_to_seconds)

    df["source_dataset"] = source_dataset

    return df.drop(columns=[
        "home_skaters", "away_skaters",
        "home_goals", "away_goals",
    ])

In [92]:
nwhl_mapped = map_legacy_columns(nwhl_df, source_dataset='NWHL')
womens_mapped = map_legacy_columns(womens_df, source_dataset='Womens')

# Combine the datasets into a single DataFrame for analysis
legacy_df = pd.concat([nwhl_mapped, womens_mapped], ignore_index=True)

In [93]:
legacy_df.head()

,game_date,home_team,away_team,period,clock,team_name,player_name,event,x,y,detail_1,detail_2,detail_3,detail_4,player_name_2,x_2,y_2,is_home,team_skaters,opponent_skaters,team_goals,opponent_goals,source_dataset
0,2021-01-23,Minnesota Whitecaps,Boston Pride,1,1200,Boston Pride,Jillian Dempsey,Faceoff Win,100,43,Backhand,NaN,NaN,NaN,Stephanie Anderson,NaN,NaN,False,5,5,0,0,NWHL
1,2021-01-23,Minnesota Whitecaps,Boston Pride,1,1198,Boston Pride,McKenna Brand,Puck Recovery,107,40,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,5,5,0,0,NWHL
2,2021-01-23,Minnesota Whitecaps,Boston Pride,1,1197,Boston Pride,McKenna Brand,Zone Entry,125,28,Carried,NaN,NaN,NaN,Maddie Rowe,NaN,NaN,False,5,5,0,0,NWHL
3,2021-01-23,Minnesota Whitecaps,Boston Pride,1,1195,Boston Pride,McKenna Brand,Shot,131,28,Snapshot,On Net,t,f,NaN,NaN,NaN,False,5,5,0,0,NWHL
4,2021-01-23,Minnesota Whitecaps,Boston Pride,1,1193,Boston Pride,Tereza Vanisova,Faceoff Win,169,21,Backhand,NaN,NaN,NaN,Stephanie Anderson,NaN,NaN,False,5,5,0,0,NWHL


#### Olympics (2022)


In [94]:
olympics_df.head()

,game_date,season_year,team_name,opp_team_name,venue,period,clock_seconds,situation_type,goals_for,goals_against,player_name,event,event_successful,x_coord,y_coord,event_type,player_name_2,x_coord_2,y_coord_2,event_detail_1,event_detail_2,event_detail_3
0,8/2/2022,2021,Olympic (Women) - Canada,Olympic (Women) - United States,away,1,1200,5 on 5,0,0,Marie-Philip Poulin,Faceoff Win,t,100,42,Backhand,Hannah Brandt,NaN,NaN,NaN,NaN,NaN
1,8/2/2022,2021,Olympic (Women) - Canada,Olympic (Women) - United States,away,1,1199,5 on 5,0,0,Jocelyne Larocque,Puck Recovery,t,86,31,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,8/2/2022,2021,Olympic (Women) - Canada,Olympic (Women) - United States,away,1,1198,5 on 5,0,0,Jocelyne Larocque,Dump In/Out,f,96,15,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,8/2/2022,2021,Olympic (Women) - Canada,Olympic (Women) - United States,away,1,1197,5 on 5,0,0,Jocelyne Larocque,Zone Entry,t,124,1,Dumped,Cayla Barnes,NaN,NaN,NaN,NaN,NaN
4,8/2/2022,2021,Olympic (Women) - United States,Olympic (Women) - Canada,home,1,1194,5 on 5,0,0,Lee Stecklein,Puck Recovery,t,6,73,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [95]:
olympics_columns = {
    "opp_team_name": "opponent_team_name",
    "goals_for": "team_goals",
    "goals_against": "opponent_goals",
    "x_coord": "x",
    "y_coord": "y",
    "x_coord_2": "x_2",
    "y_coord_2": "y_2",
    "event_type": "detail_1",
    "event_detail_1": "detail_2",
    "event_detail_2": "detail_3",
    "event_detail_3": "detail_4",
}

In [96]:
def map_olympics_columns(raw):
    df = raw.rename(columns=olympics_columns).copy()

    # Zone Entry scores use the opposite perspective in this source.
    is_entry = df["event"].eq("Zone Entry")
    df.loc[is_entry, ["team_goals", "opponent_goals"]] = (
        df.loc[is_entry, ["opponent_goals", "team_goals"]].to_numpy()
    )

    if (~df["venue"].isin(["home", "away"])).any():
        raise ValueError("Some event venues are not home or away.")

    df["is_home"] = df["venue"].eq("home").astype("boolean")
    df["home_team"] = df["team_name"].where(
        df["is_home"], df["opponent_team_name"]
    )
    df["away_team"] = df["opponent_team_name"].where(
        df["is_home"], df["team_name"]
    )

    skaters = (
        df["situation_type"]
        .astype("string")
        .str.extract(r"(?i)^\s*(\d+)\s+on\s+(\d+)\s*$")
        .astype("Int64")
    )

    if skaters.isna().any(axis=1).any():
        raise ValueError("Some situations cannot be split into skater counts.")

    df["team_skaters"] = skaters[0]
    df["opponent_skaters"] = skaters[1]

    df["source_dataset"] = "Olympics 2022"

    return df.drop(columns=[
        "venue",
        "season_year",
        "situation_type",
        "opponent_team_name",
    ])

In [97]:
olympics_df = map_olympics_columns(olympics_df)

In [98]:
olympics_df.head()


,game_date,team_name,period,clock_seconds,team_goals,opponent_goals,player_name,event,event_successful,x,y,detail_1,player_name_2,x_2,y_2,detail_2,detail_3,detail_4,is_home,home_team,away_team,team_skaters,opponent_skaters,source_dataset
0,8/2/2022,Olympic (Women) - Canada,1,1200,0,0,Marie-Philip Poulin,Faceoff Win,t,100,42,Backhand,Hannah Brandt,NaN,NaN,NaN,NaN,NaN,False,Olympic (Women) - United States,Olympic (Women) - Canada,5,5,Olympics 2022
1,8/2/2022,Olympic (Women) - Canada,1,1199,0,0,Jocelyne Larocque,Puck Recovery,t,86,31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,Olympic (Women) - United States,Olympic (Women) - Canada,5,5,Olympics 2022
2,8/2/2022,Olympic (Women) - Canada,1,1198,0,0,Jocelyne Larocque,Dump In/Out,f,96,15,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,Olympic (Women) - United States,Olympic (Women) - Canada,5,5,Olympics 2022
3,8/2/2022,Olympic (Women) - Canada,1,1197,0,0,Jocelyne Larocque,Zone Entry,t,124,1,Dumped,Cayla Barnes,NaN,NaN,NaN,NaN,NaN,False,Olympic (Women) - United States,Olympic (Women) - Canada,5,5,Olympics 2022
4,8/2/2022,Olympic (Women) - United States,1,1194,0,0,Lee Stecklein,Puck Recovery,t,6,73,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,Olympic (Women) - United States,Olympic (Women) - Canada,5,5,Olympics 2022


### Team name standardization

In [99]:
def standardize_team_names(df, date_format):
    df = df.copy()

    dates = pd.to_datetime(
        df["game_date"], format=date_format, errors="raise"
    )
    suffix = " (O" + dates.dt.strftime("%y").astype("string") + ")"
    prefix = r"^Olympic \(Women\)\s*-\s*"

    for column in ["home_team", "away_team", "team_name"]:
        names = df[column].astype("string").str.strip()
        is_olympic = names.str.contains(prefix, regex=True, na=False)

        if (is_olympic & dates.isna()).any():
            raise ValueError("Some Olympic teams have no game year.")

        df[column] = names.where(
            ~is_olympic,
            names.str.replace(prefix, "", regex=True) + suffix,
        )

    return df


legacy_df = standardize_team_names(legacy_df, "%Y-%m-%d")
olympics_df = standardize_team_names(olympics_df, "%d/%m/%Y")

## 1.3. Normalize events and outcomes

### Olympics: fix missing penalty shot

Marie-Philip Poulin scored in a penalty shot at 2:35 of February 8, 2022 game. Add a row with new event type Penalty Shot goal

In [100]:
# Confirmed missing goal: Marie-Philip Poulin's penalty shot, Feb 8, 2022.
# Insert before the USA faceoff at 2:35 in period 2; keep pre-goal scores.
game_date = pd.to_datetime(olympics_df["game_date"], format="%d/%m/%Y")
correction_game = (
    game_date.eq(pd.Timestamp("2022-02-08"))
    & olympics_df["home_team"].eq("United States (O22)")
    & olympics_df["away_team"].eq("Canada (O22)")
)
existing_goal = correction_game & olympics_df["event"].eq("Penalty Shot Goal")

if not existing_goal.any():
    anchor = (
        correction_game
        & olympics_df["period"].eq(2)
        & olympics_df["clock_seconds"].eq(155)
        & olympics_df["event"].eq("Faceoff Win")
        & olympics_df["team_name"].eq("United States (O22)")
        & olympics_df["team_goals"].eq(2)
        & olympics_df["opponent_goals"].eq(4)
    )
    positions = np.flatnonzero(anchor.to_numpy())
    if len(positions) != 1:
        raise ValueError("Could not uniquely locate the penalty-shot insertion point.")

    position = int(positions[0])
    goal = olympics_df.iloc[[position]].copy()
    goal["event"] = "Penalty Shot Goal"
    goal["team_name"] = "Canada (O22)"
    goal["is_home"] = False
    goal["player_name"] = "Marie-Philip Poulin"
    goal["player_name_2"] = pd.NA
    goal["event_successful"] = "t"
    goal["team_goals"] = 3
    goal["opponent_goals"] = 2
    goal["x"] = 100
    goal["y"] = 42

    # Carry surrounding manpower context in Canada's perspective.
    goal["team_skaters"] = olympics_df.iloc[position]["opponent_skaters"]
    goal["opponent_skaters"] = olympics_df.iloc[position]["team_skaters"]
    # No verified release location or shot details are available.
    for column in ["x_2", "y_2"]:
        goal[column] = np.nan
    for column in ["detail_1", "detail_2", "detail_3", "detail_4"]:
        goal[column] = pd.NA

    olympics_df = pd.concat([
        olympics_df.iloc[:position], goal, olympics_df.iloc[position:],
    ], ignore_index=True)

    print("Inserted the confirmed Penalty Shot Goal at period 2, 2:35.")
else:
    print("Penalty Shot Goal correction is already present.")


Inserted the confirmed Penalty Shot Goal at period 2, 2:35.


### Olympic: fix outcome of dump events

In [101]:
dump_mask = olympics_df["event"].eq("Dump In/Out")

dump_outcomes = (
    olympics_df.loc[dump_mask, "event_successful"]
    .astype("string")
    .str.strip()
    .str.lower()
    .map({
        "t": "Retained",
        "true": "Retained",
        "f": "Lost",
        "false": "Lost",
    })
)

# Missing or unrecognized outcomes remain missing.
olympics_df.loc[dump_mask, "detail_1"] = dump_outcomes

display(
    olympics_df.loc[dump_mask, "detail_1"]
    .value_counts(dropna=False)
)

detail_1
Lost        743
Retained    101
Name: count, dtype: int64

### Olympic: normalize Play
Olympic dataset does not have Incomplete Play events

Map the Play events to: Pass, Pass Reception, Incomplete Pass, and Incomplete Pass Reception

Drop second coordinates

In [102]:
flags = olympics_df["event_successful"].astype("string").str.strip().str.lower()
outcomes = {"t": True, "f": False, "true": True, "false": False}

if (flags.notna() & ~flags.isin(outcomes)).any():
    raise ValueError("Some Olympics outcome flags are not recognized.")

olympics_df["event_successful"] = flags.map(outcomes).astype("boolean")

In [103]:
df = olympics_df.copy()

# Preserve the original row and ordering through expansion.
df["source_row"] = range(len(df))
df["event_part"] = 0

is_play = df["event"].eq("Play")

if df.loc[is_play, "event_successful"].isna().any():
    raise ValueError("Some Play events have unknown outcomes.")

# Create receiver rows before changing the original Play labels.
receptions = df.loc[is_play].copy()
successful = receptions["event_successful"]

receptions["event"] = "Incomplete Pass Reception"
receptions.loc[successful, "event"] = "Pass Reception"
receptions["player_name"] = receptions["player_name_2"]
receptions["x"] = receptions["x_2"]
receptions["y"] = receptions["y_2"]
receptions["player_name_2"] = pd.NA
receptions["event_part"] = 1

# Convert the original rows into sender events.
df.loc[is_play, "event"] = "Incomplete Pass"
df.loc[is_play & df["event_successful"].fillna(False), "event"] = "Pass"

olympics_df = (
    pd.concat([df, receptions], ignore_index=True)
    .sort_values(["source_row", "event_part"], kind="stable")
    .drop(columns=["x_2", "y_2", "source_row", "event_part"])
    .reset_index(drop=True)
)

### Olympics: normalize Goal

Olympics does not have Goal event but marks successuf shots with event_successful instead. Create Goal events based on the SHot events.

In [104]:
is_goal = (
    olympics_df["event"].eq("Shot")
    & olympics_df["event_successful"]
)

olympics_df.loc[is_goal, "event"] = "Goal"

print(f"Converted {is_goal.sum()} successful shots to Goal events.")

Converted 38 successful shots to Goal events.


### Legacy: normalize Play and Incomplete Play

Map Play and Incomplete Play to:  Pass, Pass Reception, Incomplete Pass, and Incomplete Pass Reception

Drop second coordinates fields

In [105]:
df = legacy_df.copy()

df["source_row"] = range(len(df))
df["event_part"] = 0

is_play = df["event"].isin(["Play", "Incomplete Play"])

# Create actual or intended receiver rows.
receptions = df.loc[is_play].copy()
receptions["event"] = receptions["event"].map({
    "Play": "Pass Reception",
    "Incomplete Play": "Incomplete Pass Reception",
})
receptions["player_name"] = receptions["player_name_2"]
receptions["x"] = receptions["x_2"]
receptions["y"] = receptions["y_2"]
receptions["player_name_2"] = pd.NA
receptions["event_part"] = 1

# Convert the original rows into sender events.
df.loc[is_play, "event"] = df.loc[is_play, "event"].map({
    "Play": "Pass",
    "Incomplete Play": "Incomplete Pass",
})

legacy_df = (
    pd.concat([df, receptions], ignore_index=True)
    .sort_values(["source_row", "event_part"], kind="stable")
    .drop(columns=["x_2", "y_2", "source_row", "event_part"])
    .reset_index(drop=True)
)

### Legacy and Olympics: normalize Shot and Goal details

In [106]:
shot_detail_labels = {
    "detail_3": {
        "t": "Traffic",
        "f": "No traffic",
    },
    "detail_4": {
        "t": "One timer",
        "f": "",
    },
}

for df in (legacy_df, olympics_df):
    is_shot_or_goal = df["event"].isin(["Shot", "Goal"])

    for column, labels in shot_detail_labels.items():
        values = df.loc[is_shot_or_goal, column].astype("string")
        normalized = values.str.strip().str.lower()

        df.loc[is_shot_or_goal, column] = (
            normalized.map(labels).fillna(values)
        )

## 1.4. Align columns and merge

### Drop event_successful

In [107]:
olympics_df = olympics_df.drop(columns=["event_successful"])

### Parse data types

In [108]:
integer_columns = [
    "period",
    "team_goals",
    "opponent_goals",
    "team_skaters",
    "opponent_skaters",
]

string_columns = [
    "source_dataset",
    "home_team",
    "away_team",
    "team_name",
    "player_name",
    "player_name_2",
    "event",
    "detail_1",
    "detail_2",
    "detail_3",
    "detail_4",
]


def parse_data_types(df, date_format, clock_column):
    df = df.copy()

    df["game_date"] = pd.to_datetime(
        df["game_date"], format=date_format, errors="raise"
    )

    for column in integer_columns:
        df[column] = pd.to_numeric(
            df[column], errors="raise"
        ).astype("Int64")

    for column in [clock_column, "x", "y"]:
        df[column] = pd.to_numeric(
            df[column], errors="raise"
        ).astype("Float64")

    df[string_columns] = df[string_columns].astype("string")
    df["is_home"] = df["is_home"].astype("boolean")

    return df


legacy_df = parse_data_types(
    legacy_df, date_format="%Y-%m-%d", clock_column="clock"
)

olympics_df = parse_data_types(
    olympics_df, date_format="%d/%m/%Y", clock_column="clock_seconds"
)

### Align columns

In [109]:
legacy_df = legacy_df.rename(columns={"clock": "clock_seconds"}).copy()

event_columns = [
    "source_dataset",
    "game_date",
    "home_team",
    "away_team",
    "team_name",
    "is_home",
    "player_name",
    "player_name_2",
    "period",
    "clock_seconds",
    "team_goals",
    "opponent_goals",
    "team_skaters",
    "opponent_skaters",
    "event",
    "x",
    "y",
    "detail_1",
    "detail_2",
    "detail_3",
    "detail_4",
]

# Check both dataframes against the expected columns.
for name, df in [("Legacy", legacy_df), ("Olympics", olympics_df)]:
    missing = set(event_columns) - set(df.columns)
    extra = set(df.columns) - set(event_columns)

    if not df.columns.is_unique:
        raise ValueError(f"{name} has duplicate column names.")

    if missing or extra:
        raise ValueError(
            f"{name}: missing={sorted(missing)}, extra={sorted(extra)}"
        )

# Align column order.
legacy_df = legacy_df[event_columns]
olympics_df = olympics_df[event_columns]

# Check matching data types.
type_comparison = pd.DataFrame({
    "legacy": legacy_df.dtypes,
    "olympics": olympics_df.dtypes,
})
type_comparison["match"] = (
    type_comparison["legacy"] == type_comparison["olympics"]
)

display(type_comparison)

if not type_comparison["match"].all():
    raise TypeError("Data types differ between Legacy and Olympics.")

print("Column names, order, and data types match.")

,legacy,olympics,match
source_dataset,string[python],string[python],True
game_date,datetime64[ns],datetime64[ns],True
home_team,string[python],string[python],True
away_team,string[python],string[python],True
team_name,string[python],string[python],True
is_home,boolean,boolean,True
player_name,string[python],string[python],True
player_name_2,string[python],string[python],True
period,Int64,Int64,True
clock_seconds,Float64,Float64,True


Column names, order, and data types match.


### Merge dataframes

In [110]:
events_df = pd.concat(
    [legacy_df, olympics_df],
    ignore_index=True,
)

assert len(events_df) == len(legacy_df) + len(olympics_df)

print(f"Combined {len(events_df):,} events.")
display(events_df.head())

Combined 86,670 events.


,source_dataset,game_date,home_team,away_team,team_name,is_home,player_name,player_name_2,period,clock_seconds,team_goals,opponent_goals,team_skaters,opponent_skaters,event,x,y,detail_1,detail_2,detail_3,detail_4
0,NWHL,2021-01-23,Minnesota Whitecaps,Boston Pride,Boston Pride,False,Jillian Dempsey,Stephanie Anderson,1,1200.0,0,0,5,5,Faceoff Win,100.0,43.0,Backhand,<NA>,<NA>,<NA>
1,NWHL,2021-01-23,Minnesota Whitecaps,Boston Pride,Boston Pride,False,McKenna Brand,<NA>,1,1198.0,0,0,5,5,Puck Recovery,107.0,40.0,<NA>,<NA>,<NA>,<NA>
2,NWHL,2021-01-23,Minnesota Whitecaps,Boston Pride,Boston Pride,False,McKenna Brand,Maddie Rowe,1,1197.0,0,0,5,5,Zone Entry,125.0,28.0,Carried,<NA>,<NA>,<NA>
3,NWHL,2021-01-23,Minnesota Whitecaps,Boston Pride,Boston Pride,False,McKenna Brand,<NA>,1,1195.0,0,0,5,5,Shot,131.0,28.0,Snapshot,On Net,Traffic,
4,NWHL,2021-01-23,Minnesota Whitecaps,Boston Pride,Boston Pride,False,Tereza Vanisova,Stephanie Anderson,1,1193.0,0,0,5,5,Faceoff Win,169.0,21.0,Backhand,<NA>,<NA>,<NA>


## 1.5. Build team, player, game, events records

### Teams

In [111]:
# Teams: include both participants from every game.
teams_df = (
    pd.concat([
        events_df[["source_dataset", "home_team"]]
        .rename(columns={"home_team": "team_name"}),

        events_df[["source_dataset", "away_team"]]
        .rename(columns={"away_team": "team_name"}),
    ], ignore_index=True)
    .dropna(subset=["team_name"])
    .drop_duplicates()
    .sort_values(["source_dataset", "team_name"])
    .reset_index(drop=True)
)

teams_df.insert(0, "team_id", range(1, len(teams_df) + 1))

print(f"Identified {len(teams_df):,} unique teams.")
display(teams_df.head())


Identified 20 unique teams.


,team_id,source_dataset,team_name
0,1,NWHL,Boston Pride
1,2,NWHL,Buffalo Beauts
2,3,NWHL,Connecticut Whale
3,4,NWHL,Metropolitan Riveters
4,5,NWHL,Minnesota Whitecaps


### Players

In [112]:
primary_players = (
    events_df[["source_dataset", "team_name", "player_name"]]
    .dropna(subset=["player_name"])
    .drop_duplicates()
)

secondary = events_df.loc[
    events_df["player_name_2"].notna(),
    ["source_dataset", "home_team", "away_team",
     "team_name", "event", "player_name_2"],
].copy()

same_team_events = [
    "Pass", "Incomplete Pass",
    "Pass Reception", "Incomplete Pass Reception",
]

opponent_team_events = [
    "Zone Entry",
    "Faceoff Win",
    "Penalty Taken",
]

supported_events = same_team_events + opponent_team_events

if (~secondary["event"].isin(supported_events)).any():
    display(secondary.loc[~secondary["event"].isin(supported_events)])
    raise ValueError("Unexpected event with a secondary player.")

valid_team = (
    secondary["team_name"].eq(secondary["home_team"])
    | secondary["team_name"].eq(secondary["away_team"])
)

if not valid_team.all():
    raise ValueError("Some event teams do not match either game team.")

opponent_team = secondary["away_team"].where(
    secondary["team_name"].eq(secondary["home_team"]),
    secondary["home_team"],
)

secondary["team_name"] = secondary["team_name"].where(
    secondary["event"].isin(same_team_events),
    opponent_team,
)

secondary_players = (
    secondary[["source_dataset", "team_name", "player_name_2"]]
    .rename(columns={"player_name_2": "player_name"})
)

players_df = (
    pd.concat([primary_players, secondary_players], ignore_index=True)
    .drop_duplicates()
    .merge(
        teams_df,
        on=["source_dataset", "team_name"],
        how="left",
        validate="many_to_one",
    )
)

if players_df["team_id"].isna().any():
    raise ValueError("Some players could not be linked to a team.")

players_df = (
    players_df
    .sort_values(["source_dataset", "team_id", "player_name"])
    .reset_index(drop=True)
)

players_df.insert(0, "player_id", range(1, len(players_df) + 1))

players_df = players_df[
    ["player_id", "player_name", "team_id", "source_dataset"]
]
players_df[["player_id", "team_id"]] = (
    players_df[["player_id", "team_id"]].astype("Int64")
)

print(f"Identified {len(players_df):,} unique players.")
display(players_df.head())

Identified 435 unique players.


,player_id,player_name,team_id,source_dataset
0,1,Briana Mastel,1,NWHL
1,2,Carlee Turner,1,NWHL
2,3,Christina Putigna,1,NWHL
3,4,Jenna Rheault,1,NWHL
4,5,Jillian Dempsey,1,NWHL


### Games

In [113]:
games_df = (
    events_df[
        ["source_dataset", "game_date", "home_team", "away_team"]
    ]
    .drop_duplicates()
    .merge(
        teams_df.rename(columns={
            "team_name": "home_team",
            "team_id": "home_team_id",
        }),
        on=["source_dataset", "home_team"],
        how="left",
        validate="many_to_one",
    )
    .merge(
        teams_df.rename(columns={
            "team_name": "away_team",
            "team_id": "away_team_id",
        }),
        on=["source_dataset", "away_team"],
        how="left",
        validate="many_to_one",
    )
)

if games_df[["home_team_id", "away_team_id"]].isna().any().any():
    raise ValueError("Some games could not be linked to both teams.")

if games_df["home_team_id"].eq(games_df["away_team_id"]).any():
    raise ValueError("Some games have the same home and away team.")

games_df = (
    games_df
    .sort_values(["source_dataset", "game_date", "home_team", "away_team"])
    .reset_index(drop=True)
)

games_df.insert(0, "game_id", range(1, len(games_df) + 1))

games_df = games_df[
    ["game_id", "game_date", "home_team_id", "away_team_id", "source_dataset"]
]

id_columns = ["game_id", "home_team_id", "away_team_id"]
games_df[id_columns] = games_df[id_columns].astype("Int64")

print(f"Identified {len(games_df):,} unique games.")
display(games_df.head())

Identified 34 unique games.


,game_id,game_date,home_team_id,away_team_id,source_dataset
0,1,2021-01-23,2,3,NWHL
1,2,2021-01-23,4,6,NWHL
2,3,2021-01-23,5,1,NWHL
3,4,2021-01-24,1,2,NWHL
4,5,2021-01-24,3,4,NWHL


### Event types


In [114]:
event_types_df = (
    events_df[["event"]]
    .drop_duplicates()
    .sort_values("event")
    .reset_index(drop=True)
)

if event_types_df["event"].isna().any():
    raise ValueError("Some events have no event label.")

event_types_df.insert(0, "event_type_id", range(1, len(event_types_df) + 1))
event_types_df["event_type_id"] = event_types_df["event_type_id"].astype("Int64")

assert event_types_df["event"].is_unique
assert events_df["event"].isin(event_types_df["event"]).all()

print(f"Identified {len(event_types_df):,} unique event types.")
display(event_types_df)


Identified 13 unique event types.


,event_type_id,event
0,1,Dump In/Out
1,2,Faceoff Win
2,3,Goal
3,4,Incomplete Pass
4,5,Incomplete Pass Reception
5,6,Pass
6,7,Pass Reception
7,8,Penalty Shot Goal
8,9,Penalty Taken
9,10,Puck Recovery


### Events

In [115]:
events_table_df = events_df.copy()
events_table_df.insert(0, "event_id", range(1, len(events_table_df) + 1))

# Link event, home, and away teams.
for name_column, id_column in [
    ("team_name", "team_id"),
    ("home_team", "home_team_id"),
    ("away_team", "away_team_id"),
]:
    events_table_df = events_table_df.merge(
        teams_df.rename(columns={
            "team_name": name_column,
            "team_id": id_column,
        }),
        on=["source_dataset", name_column],
        how="left",
        validate="many_to_one",
    )

# Identify the other participant from the event team's perspective.
events_table_df["opponent_team_id"] = events_table_df["away_team_id"].where(
    events_table_df["team_id"].eq(events_table_df["home_team_id"]),
    events_table_df["home_team_id"],
)

# Link games.
events_table_df = events_table_df.merge(
    games_df,
    on=[
        "source_dataset", "game_date",
        "home_team_id", "away_team_id",
    ],
    how="left",
    validate="many_to_one",
)

# Link primary players.
events_table_df = events_table_df.merge(
    players_df,
    on=["source_dataset", "team_id", "player_name"],
    how="left",
    validate="many_to_one",
)

# Resolve secondary players in a separate lookup.
secondary_lookup = events_table_df.loc[
    events_table_df["player_name_2"].notna(),
    ["event_id", "source_dataset", "event", "player_name_2",
     "team_id", "home_team_id", "away_team_id"],
].copy()

same_team_events = ["Play", "Incomplete Play", "Pass", "Incomplete Pass"]
opponent_team_events = ["Zone Entry", "Faceoff Win", "Penalty Taken"]

if (~secondary_lookup["event"].isin(
    same_team_events + opponent_team_events
)).any():
    raise ValueError("Unexpected event with a secondary player.")

opponent_id = secondary_lookup["away_team_id"].where(
    secondary_lookup["team_id"].eq(secondary_lookup["home_team_id"]),
    secondary_lookup["home_team_id"],
)

secondary_lookup["team_id"] = secondary_lookup["team_id"].where(
    secondary_lookup["event"].isin(same_team_events),
    opponent_id,
)

secondary_lookup = secondary_lookup.merge(
    players_df.rename(columns={
        "player_name": "player_name_2",
        "player_id": "player_2_id",
    }),
    on=["source_dataset", "team_id", "player_name_2"],
    how="left",
    validate="many_to_one",
)

if secondary_lookup["player_2_id"].isna().any():
    display(secondary_lookup.loc[secondary_lookup["player_2_id"].isna()])
    raise ValueError("Some secondary players could not be linked.")

events_table_df = events_table_df.merge(
    secondary_lookup[["event_id", "player_2_id"]],
    on="event_id",
    how="left",
    validate="one_to_one",
)

# Validate required links.
if events_table_df[["game_id", "team_id", "opponent_team_id"]].isna().any().any():
    raise ValueError("Some events could not be linked to a game or team.")

missing_primary = (
    events_table_df["player_name"].notna()
    & events_table_df["player_id"].isna()
)
if missing_primary.any():
    raise ValueError("Some primary players could not be linked.")

# Select the final EDA schema.
event_columns = [
    "event_id", "game_id", "team_id", "opponent_team_id", "player_id", "player_2_id",
    "period", "clock_seconds",
    "team_goals", "opponent_goals",
    "team_skaters", "opponent_skaters",
    "event", "x", "y",
    "detail_1", "detail_2", "detail_3", "detail_4",
    "is_home", "source_dataset",
]

events_table_df = (
    events_table_df[event_columns]
    .sort_values("event_id")
    .reset_index(drop=True)
)

id_columns = [
    "event_id", "game_id", "team_id", "opponent_team_id", "player_id", "player_2_id",
]
events_table_df[id_columns] = events_table_df[id_columns].astype("Int64")

assert len(events_table_df) == len(events_df)
assert events_table_df["event_id"].is_unique

print(f"Identified {len(events_table_df):,} unique events.")
display(events_table_df.head())

Identified 86,670 unique events.


,event_id,game_id,team_id,opponent_team_id,player_id,player_2_id,period,clock_seconds,team_goals,opponent_goals,team_skaters,opponent_skaters,event,x,y,detail_1,detail_2,detail_3,detail_4,is_home,source_dataset
0,1,3,1,5,5,104,1,1200.0,0,0,5,5,Faceoff Win,100.0,43.0,Backhand,<NA>,<NA>,<NA>,False,NWHL
1,2,3,1,5,12,<NA>,1,1198.0,0,0,5,5,Puck Recovery,107.0,40.0,<NA>,<NA>,<NA>,<NA>,False,NWHL
2,3,3,1,5,12,98,1,1197.0,0,0,5,5,Zone Entry,125.0,28.0,Carried,<NA>,<NA>,<NA>,False,NWHL
3,4,3,1,5,12,<NA>,1,1195.0,0,0,5,5,Shot,131.0,28.0,Snapshot,On Net,Traffic,,False,NWHL
4,5,3,1,5,19,104,1,1193.0,0,0,5,5,Faceoff Win,169.0,21.0,Backhand,<NA>,<NA>,<NA>,False,NWHL


## 1.6. Validate processed tables

### Table structure and keys

Check required values, primary keys, logical types, and unique team, player, game, and event-type records. Missing optional values remain allowed.


In [116]:
tables = {
    "teams": (teams_df, "team_id", ["team_name", "source_dataset"]),
    "players": (players_df, "player_id", ["player_name", "team_id", "source_dataset"]),
    "games": (games_df, "game_id", ["game_date", "home_team_id", "away_team_id", "source_dataset"]),
    "event_types": (event_types_df, "event_type_id", ["event"]),
    "events": (events_table_df, "event_id", [
        "game_id", "team_id", "opponent_team_id", "period", "team_skaters", "opponent_skaters",
        "event", "is_home", "source_dataset",
    ]),
}

validation_results = []


def check(name, passed):
    validation_results.append({"check": name, "passed": bool(passed)})


for name, (df, key, required) in tables.items():
    check(f"{name}: unique columns", df.columns.is_unique)
    check(f"{name}: required values", df[[key] + required].notna().all().all())
    check(f"{name}: unique primary key", df[key].is_unique)
    check(f"{name}: positive primary key", df[key].gt(0).all())
    for column in df.columns:
        if column.endswith("_id") or column in integer_columns:
            check(f"{name}: {column} integer type", pd.api.types.is_integer_dtype(df[column]))
        elif column in ["clock_seconds", "x", "y"]:
            check(f"{name}: {column} numeric type", pd.api.types.is_numeric_dtype(df[column]))
        elif column == "is_home":
            check(f"{name}: Boolean type", pd.api.types.is_bool_dtype(df[column]))
        elif column == "game_date":
            check(f"{name}: date type", pd.api.types.is_datetime64_any_dtype(df[column]))
        else:
            check(f"{name}: {column} string type", isinstance(df[column].dtype, pd.StringDtype))
    for column in [key] + required:
        if isinstance(df[column].dtype, pd.StringDtype):
            check(f"{name}: {column} nonblank", df[column].str.strip().ne("").fillna(False).all())

for name, df, columns in [
    ("teams", teams_df, ["source_dataset", "team_name"]),
    ("players", players_df, ["source_dataset", "team_id", "player_name"]),
    ("games", games_df, ["source_dataset", "game_date", "home_team_id", "away_team_id"]),
    ("event_types", event_types_df, ["event"]),
]:
    check(f"{name}: unique records", not df.duplicated(columns).any())


### Relationships and player attribution

Verify foreign keys, source consistency, participating teams, home attribution, and the event-specific secondary-player rules.


In [117]:
team_sources = teams_df.set_index("team_id")["source_dataset"]
player_lookup = players_df.set_index("player_id")
game_lookup = games_df.set_index("game_id")
e = events_table_df

check("players: valid team links", players_df["team_id"].isin(teams_df["team_id"]).all())
check("players: source matches team", players_df["source_dataset"].eq(players_df["team_id"].map(team_sources)).all())

for column in ["home_team_id", "away_team_id"]:
    check(f"games: valid {column}", games_df[column].isin(teams_df["team_id"]).all())
    check(f"games: {column} source", games_df["source_dataset"].eq(games_df[column].map(team_sources)).all())
check("games: distinct teams", games_df["home_team_id"].ne(games_df["away_team_id"]).all())

check("events: valid game links", e["game_id"].isin(games_df["game_id"]).all())
check("events: valid team links", e["team_id"].isin(teams_df["team_id"]).all())
check("events: game source", e["source_dataset"].eq(e["game_id"].map(game_lookup["source_dataset"])).all())
check("events: team source", e["source_dataset"].eq(e["team_id"].map(team_sources)).all())
check("events: valid opponent links", e["opponent_team_id"].isin(teams_df["team_id"]).all())
check("events: opponent source", e["source_dataset"].eq(e["opponent_team_id"].map(team_sources)).all())
check("events: distinct opponent", e["team_id"].ne(e["opponent_team_id"]).all())
check("events: known event labels", e["event"].isin(event_types_df["event"]).all())

home_ids = e["game_id"].map(game_lookup["home_team_id"])
away_ids = e["game_id"].map(game_lookup["away_team_id"])
expected_team = home_ids.where(e["is_home"], away_ids)
check("events: home attribution", e["team_id"].eq(expected_team).all())

for column in ["player_id", "player_2_id"]:
    present = e[column].notna()
    check(f"events: valid {column}", e.loc[present, column].isin(players_df["player_id"]).all())
    check(f"events: {column} source", e.loc[present, "source_dataset"].eq(e.loc[present, column].map(player_lookup["source_dataset"])).all())

primary = e["player_id"].notna()
check("events: primary player team", e.loc[primary, "player_id"].map(player_lookup["team_id"]).eq(e.loc[primary, "team_id"]).all())

secondary = e["player_2_id"].notna()
same_team = e["event"].isin(["Pass", "Incomplete Pass"])
opponent_event = e["event"].isin(["Zone Entry", "Faceoff Win", "Penalty Taken"])
check("events: secondary event rules", (same_team | opponent_event).loc[secondary].all())
opponent_ids = away_ids.where(e["is_home"], home_ids)
check("events: opponent attribution", e["opponent_team_id"].eq(opponent_ids).all())
expected_secondary_team = e["team_id"].where(same_team, opponent_ids)
check("events: secondary player team", e.loc[secondary, "player_2_id"].map(player_lookup["team_id"]).eq(expected_secondary_team.loc[secondary]).all())


### Event counts, ranges, and ordering

Validation


In [118]:
expected_counts = {
    "NWHL": len(nwhl_df) + nwhl_df["Event"].isin(["Play", "Incomplete Play"]).sum(),
    "Womens": len(womens_df) + womens_df["Event"].isin(["Play", "Incomplete Play"]).sum(),
    "Olympics 2022": len(datasets["Olympics"]) + datasets["Olympics"]["event"].eq("Play").sum() + 1,  # Confirmed missing penalty-shot goal
}
actual_counts = e.groupby("source_dataset").size()
check("events: expected sources", set(actual_counts.index) == set(expected_counts))
for source, expected in expected_counts.items():
    check(f"{source}: expanded row count", actual_counts.get(source, 0) == expected)
check("events: table row count", len(e) == len(events_df))
check("events: no unexpanded plays", not e["event"].isin(["Play", "Incomplete Play"]).any())

check("events: positive periods", e["period"].ge(1).all())
check("events: finite nonnegative clocks", e["clock_seconds"].dropna().between(0, 9999.999999).all())
for column in ["team_goals", "opponent_goals", "team_skaters", "opponent_skaters"]:
    check(f"events: nonnegative {column}", e[column].dropna().ge(0).all())
for column, upper in [("x", 200), ("y", 85)]:
    check(f"events: {column} bounds", e[column].dropna().between(0, upper).all())

ordered = e.sort_values("event_id")
period_change = ordered.groupby("game_id", sort=False)["period"].diff()
clock_change = ordered.groupby(["game_id", "period"], sort=False)["clock_seconds"].diff()
check("events: period order", not period_change.lt(0).any())
check("events: countdown clock order", not clock_change.gt(0).any())

shots = e.loc[e["event"].isin(["Shot", "Goal"])]
check("shots: traffic labels", shots["detail_3"].dropna().isin(["Traffic", "No traffic"]).all())
check("shots: one-timer labels", shots["detail_4"].dropna().isin(["One timer", ""]).all())

validation_report = pd.DataFrame(validation_results)
display(validation_report)
display(pd.DataFrame({"expected": expected_counts, "actual": actual_counts}))

failed = validation_report.loc[~validation_report["passed"]]
if not failed.empty:
    display(failed)
    raise ValueError(f"{len(failed)} validation checks failed; resolve before export.")

print(f"All {len(validation_report)} validation checks passed.")


,check,passed
0,teams: unique columns,True
1,teams: required values,True
2,teams: unique primary key,True
3,teams: positive primary key,True
4,teams: team_id integer type,True
...,...,...
104,events: y bounds,True
105,events: period order,True
106,events: countdown clock order,True
107,shots: traffic labels,True


,expected,actual
NWHL,37561,37561
Olympics 2022,15002,15002
Womens,34107,34107


All 109 validation checks passed.


### Olympics score consistency

Validate corrected scores against cumulative `Goal` and `Penalty Shot Goal` events within each game. Accept scores before or after the goal on goal rows; other rows must agree with the running totals. Missing scores fail this check.


In [119]:
# Verify corrected Olympics scores independently of the raw diagnostics.
olympics_score_check = (
    events_table_df.loc[events_table_df["source_dataset"].eq("Olympics 2022")]
    .sort_values("event_id")
    .copy()
)
score_check = olympics_score_check
score_check["home_score"] = score_check["team_goals"].where(
    score_check["is_home"], score_check["opponent_goals"]
)
score_check["away_score"] = score_check["opponent_goals"].where(
    score_check["is_home"], score_check["team_goals"]
)

is_goal = score_check["event"].isin(["Goal", "Penalty Shot Goal"])
for side, team_mask in [
    ("home", score_check["is_home"]),
    ("away", ~score_check["is_home"]),
]:
    score_check[f"{side}_goal"] = (is_goal & team_mask).astype(int)
    score_check[f"{side}_goals_after"] = (
        score_check.groupby("game_id", sort=False)[f"{side}_goal"].cumsum()
    )
    score_check[f"{side}_goals_before"] = (
        score_check[f"{side}_goals_after"] - score_check[f"{side}_goal"]
    )

matches_before = (
    score_check["home_score"].eq(score_check["home_goals_before"])
    & score_check["away_score"].eq(score_check["away_goals_before"])
)
matches_after = (
    score_check["home_score"].eq(score_check["home_goals_after"])
    & score_check["away_score"].eq(score_check["away_goals_after"])
)
score_check["score_mismatch"] = ~(matches_before | matches_after).fillna(False)
score_check["score_decrease"] = (
    score_check.groupby("game_id", sort=False)[["home_score", "away_score"]]
    .diff()
    .lt(0)
    .any(axis=1)
)

olympics_score_report = score_check.groupby("game_id", sort=False).agg(
    events=("event_id", "size"),
    goals=("home_goal", "sum"),
    away_goals=("away_goal", "sum"),
    score_mismatches=("score_mismatch", "sum"),
    score_decreases=("score_decrease", "sum"),
).rename(columns={"goals": "home_goals"})

display(olympics_score_report)
score_issues = score_check.loc[
    score_check["score_mismatch"] | score_check["score_decrease"]
]
if not score_issues.empty:
    display(score_issues[[
        "event_id", "game_id", "period", "clock_seconds", "event", "is_home",
        "home_score", "away_score", "home_goals_before", "away_goals_before",
        "home_goals_after", "away_goals_after", "score_mismatch", "score_decrease",
    ]])
    raise ValueError(f"{len(score_issues)} Olympics events have inconsistent scores.")

if score_check.empty:
    raise ValueError("No Olympics events were available for score validation.")

print("Olympics scores match cumulative goals with no score decreases.")


,events,home_goals,away_goals,score_mismatches,score_decreases
game_id,,,,,
17,2544,2,4,0,0
16,2467,5,0,0,0
18,2520,2,4,0,0
19,2507,10,3,0,0
20,2494,1,4,0,0
21,2470,4,0,0,0


Olympics scores match cumulative goals with no score decreases.


## 1.7. Export processed files

In [120]:
PROCESSED_DATA_DIR = Path(ROOT_DIR) / "data" / "processed"
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

# Match the SQL clock precision without changing the working events table.
events_export_df = events_table_df.copy()
events_export_df["clock_seconds"] = events_export_df["clock_seconds"].round(6)

export_tables = {
    "teams": teams_df,
    "players": players_df,
    "games": games_df,
    "event_types": event_types_df,
    "events": events_export_df,
}

export_summary = []
for name, df in export_tables.items():
    path = PROCESSED_DATA_DIR / f"{name}.parquet"
    df.to_parquet(path, engine="pyarrow", compression="snappy", index=False)

    saved = pd.read_parquet(path, engine="pyarrow")
    pd.testing.assert_frame_equal(
        saved, df.reset_index(drop=True), check_exact=True
    )

    export_summary.append({
        "table": name,
        "rows": len(saved),
        "columns": len(saved.columns),
        "file": str(path),
        "size_kb": round(path.stat().st_size / 1024, 1),
    })

print("Exported and verified all five Parquet files.")
display(pd.DataFrame(export_summary))


Exported and verified all five Parquet files.


,table,rows,columns,file,size_kb
0,teams,20,3,..\data\processed\teams.parquet,2.7
1,players,435,4,..\data\processed\players.parquet,10.1
2,games,34,5,..\data\processed\games.parquet,4.0
3,event_types,13,2,..\data\processed\event_types.parquet,1.9
4,events,86670,21,..\data\processed\events.parquet,1156.3
